# Task 3: Python and Research Programming Basics

Build a small experiment-analysis pipeline: parse messy records, validate inputs, compare matched runs, and produce a reproducible report. Sections 1–4 are warm-ups; Sections 5–7 require combining these skills. Allow roughly 4–6 hours.

## How to use this notebook
Run cells in order using Python 3. Only the Python standard library is needed; no GPU, simulator, API key, or paid service is required.

For **Predict** questions, write your answer before running the example. Wrong predictions are fine: explain what changed in your understanding. For **Exercises**, replace `None` or `pass` with your solution, then run the checks. Checks cover examples and edge cases; add at least two meaningful checks of your own in Section 7. Avoid hardcoding the example results.

## Submission
Upload the completed notebook with outputs and written responses to Canvas. Restart the kernel and run all cells before submitting. Include a short reflection: what did you learn, what was difficult, and what question remains?

## 1. Values, lists, and indexing
A list stores an ordered sequence. Indexing starts at zero; a slice excludes its ending index.

**Predict:** What do the three printed expressions return? Write your prediction here.

In [ ]:
scores = [2, 4, 6, 8]
print(scores[0])
print(scores[-1])
print(scores[1:3])

### Exercise 1
Select the last two scores and calculate the mean of all scores. Use list operations rather than typing the answers.

In [ ]:
last_two = None
mean_score = None

In [ ]:
assert last_two == [6, 8]
assert mean_score == 5
print("Checks passed")

## 2. Loops, dictionaries, and functions
A dictionary associates keys with values. A function gives a reusable operation a name. `return` sends its result to the caller.

**Predict:** Which record passes the condition?

In [ ]:
records = [{"name": "a", "score": 3}, {"name": "b", "score": 7}]
for record in records:
    if record["score"] >= 5:
        print(record["name"])

### Exercise 2
Implement `select_names(records, minimum)` to return names whose scores are at least the minimum, in input order. Return an empty list when none qualify.

In [ ]:
def select_names(records, minimum):
    pass

In [ ]:
assert select_names(records, 5) == ["b"]
assert select_names(records, 3) == ["a", "b"]
assert select_names([], 5) == []
assert select_names(records, 10) == []
print("Checks passed")

## 3. Debugging and edge cases
Read the final line of a traceback first, then inspect the values that caused it. An empty dataset needs an explicit policy.

**Predict:** What exception would `sum([]) / len([])` raise?

In [ ]:
try:
    print(sum([]) / len([]))
except ZeroDivisionError as error:
    print(type(error).__name__, str(error))

### Exercise 3
Write `average(values)`. Return the arithmetic mean for nonempty input and raise `ValueError` with a useful message for empty input. Explain why returning zero for empty input could mislead a researcher.

In [ ]:
def average(values):
    pass

In [ ]:
assert average([2, 4, 9]) == 5
assert average([-2, 2]) == 0
try:
    average([])
except ValueError as error:
    assert str(error)
else:
    raise AssertionError("Empty input must raise ValueError")
print("Checks passed")

## 4. A small reproducible analysis
CSV is a common table format. Each row below describes one experimental run. `DictReader` reads values as strings, so convert numbers explicitly.

**Predict:** What is the type of the first score?

In [ ]:
import csv
import io

data = "method,score\nbaseline,2\nbaseline,4\nnew,5\nnew,7\n"
rows = list(csv.DictReader(io.StringIO(data)))
print(rows)
print(type(rows[0]["score"]).__name__)

### Exercise 4
Group numeric scores by method and calculate the mean for each method using your `average` function. Then write 2–3 sentences explaining why two runs per method are insufficient to establish general superiority.

In [ ]:
grouped = {}
# Populate grouped from rows.
summary = None

In [ ]:
assert grouped == {"baseline": [2.0, 4.0], "new": [5.0, 7.0]}
assert summary == {"baseline": 3.0, "new": 6.0}
print("Checks passed")

## 5. Parse and validate experimental records
Real experiment logs contain missing fields, failed runs, and repeated entries. A reliable analysis makes these conditions explicit before computing a mean.

`csv.DictReader` yields strings. Strip surrounding whitespace, convert numeric fields, and validate the values. `math.isfinite` rejects NaN and infinity, which otherwise can silently corrupt a summary. Catch expected conversion errors and attach a row number rather than swallowing every exception.

**Predict:** Does `float("nan")` raise an exception? Why might checking only whether a score is between 0 and 1 be insufficient if written as `score < 0 or score > 1`?


In [ ]:
import math

print(float("nan"))
print(math.isfinite(float("nan")))

experiment_csv = """method,seed,score,status
baseline,0,0.60,ok
candidate,0,0.70,ok
baseline,1,0.80,ok
candidate,1,0.75,ok
baseline,2,0.50,ok
candidate,2,,failed
candidate,3,0.95,ok
candidate,4,nan,ok
baseline,0,0.99,ok
candidate,5,1.20,ok
"""


### Exercise 5: Validation with an audit trail
Implement `parse_runs(text)` returning `(accepted, rejected)`, both lists in input order.

- Require the header fields `method`, `seed`, `score`, and `status`; missing required columns raise `ValueError` even if there are no data rows. Extra columns may be ignored.
- Accept only nonempty method names, nonnegative integer seeds, status `ok`, and finite numeric scores in [0, 1]. Strip whitespace in each field. A missing field value is invalid.
- A `(method, seed)` pair may be accepted only once. Reject later valid duplicates; an invalid row must not reserve its key.
- Accepted entries are dictionaries with exactly `method` (string), `seed` (integer), and `score` (float).
- For each rejected data row, record `{"row": line_number, "reason": nonempty_string}`. Count the header as line 1; assume one physical line per record and no blank lines. If several rules fail, reporting one reason is enough.
- Do not mutate global data or print errors instead of returning them.

Before coding, describe the order in which you will validate a row. Explain why a failed run must not become a score of zero.


In [ ]:
def parse_runs(text):
    pass


In [ ]:
accepted, rejected = parse_runs(experiment_csv)
assert len(accepted) == 6
assert [row["row"] for row in rejected] == [7, 9, 10, 11]
assert all(isinstance(row["reason"], str) and row["reason"] for row in rejected)
assert accepted[0] == {"method": "baseline", "seed": 0, "score": 0.6}
assert accepted[-1] == {"method": "candidate", "seed": 3, "score": 0.95}
assert all(set(row) == {"method", "seed", "score"} for row in accepted)
assert all(type(row["seed"]) is int and type(row["score"]) is float for row in accepted)
assert parse_runs("method,seed,score,status\n") == ([], [])
clean, errors = parse_runs("method,seed,score,status\n a , 2 , 0.4 , ok \na,3,,failed\na,3,0.7,ok\n")
assert clean == [{"method": "a", "seed": 2, "score": 0.4}, {"method": "a", "seed": 3, "score": 0.7}]
assert [row["row"] for row in errors] == [3]
for invalid_header in ("method,score,status\n", ""):
    try:
        parse_runs(invalid_header)
    except ValueError:
        pass
    else:
        raise AssertionError("Missing required columns must raise ValueError")
print("Checks passed")


## 6. Compare matched runs
A higher overall mean can reflect different sets of experimental conditions. Here, a seed identifies a shared condition across methods. Compare scores only where both methods have a valid run; report what you excluded. In a real study, verify that a shared seed actually represents a meaningful match.

A dictionary keyed by seed makes matching easier. Sort seed IDs so your report has deterministic ordering, independent of CSV row order.

**Predict:** Does the candidate's advantage increase or decrease if its unmatched score of 0.95 is included in its mean? Calculate both method means from all accepted rows before computing matched differences.


In [ ]:
baseline_by_seed = {r["seed"]: r["score"] for r in accepted if r["method"] == "baseline"}
candidate_by_seed = {r["seed"]: r["score"] for r in accepted if r["method"] == "candidate"}
print("Baseline seeds:", sorted(baseline_by_seed))
print("Candidate seeds:", sorted(candidate_by_seed))


### Exercise 6: Paired comparison
Implement `compare_runs(runs, baseline, candidate)`. Inputs are accepted records from Exercise 5; assume unique `(method, seed)` keys. Ignore other methods. Require different method names and at least one shared seed; otherwise raise `ValueError`.

Return a dictionary with exactly these fields:

| Field | Meaning |
|---|---|
| `seeds` | Sorted shared seed IDs |
| `differences` | Candidate minus baseline score, in that order |
| `mean_difference` | Arithmetic mean of the paired differences |
| `wins` | Number of differences strictly greater than zero |
| `baseline_only` | Sorted seeds present only for baseline |
| `candidate_only` | Sorted seeds present only for candidate |

Do not modify the input. Reuse your `average` function. Explain why excluding unmatched runs should be disclosed, especially when missing runs may reflect failures.


In [ ]:
def compare_runs(runs, baseline, candidate):
    pass


In [ ]:
import copy
before = copy.deepcopy(accepted)
comparison = compare_runs(accepted, "baseline", "candidate")
assert set(comparison) == {"seeds", "differences", "mean_difference", "wins", "baseline_only", "candidate_only"}
assert comparison["seeds"] == [0, 1]
assert all(abs(a-b) < 1e-12 for a,b in zip(comparison["differences"], [0.10, -0.05]))
assert len(comparison["differences"]) == 2
assert abs(comparison["mean_difference"] - 0.025) < 1e-12
assert comparison["wins"] == 1
assert comparison["baseline_only"] == [2]
assert comparison["candidate_only"] == [3]
assert accepted == before
assert compare_runs(list(reversed(accepted)), "baseline", "candidate") == comparison
for runs, left, right in (([], "a", "b"), (accepted, "baseline", "missing"), (accepted, "baseline", "baseline")):
    try:
        compare_runs(runs, left, right)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid comparison must raise ValueError")
print("Checks passed")


## 7. Debug and assemble the pipeline
Mutable objects can accidentally share state. Run this example and explain why appending to one group also changes the other. Then describe how you would construct independent lists.

**Predict:** What will `broken_groups` contain?


In [ ]:
shared_scores = []
broken_groups = dict.fromkeys(["baseline", "candidate"], shared_scores)
broken_groups["baseline"].append(0.6)
print(broken_groups)


### Exercise 7: A reproducible report
Implement `build_report(text, baseline, candidate)` using your previous functions. Return `{"accepted_count": integer, "rejected": rejection_list, "comparison": comparison_dictionary}`. Let validation and comparison errors propagate to the caller.

Serialize the result to `report_json` using `json.dumps` with `sort_keys=True`, `indent=2`, and `allow_nan=False`. This produces a portable report without writing a file.

Add at least two checks covering cases not already tested above. Include an all-tied comparison and a malformed numeric field. Explain what failure each check would catch. Finish with a 150–200 word interpretation of the report: distinguish measured improvement, missing evidence, and a justified next experiment. Do not claim statistical significance from two matched runs.


In [ ]:
import json

def build_report(text, baseline, candidate):
    pass

report = build_report(experiment_csv, "baseline", "candidate")
report_json = None
# Add your own checks below.


In [ ]:
assert report["accepted_count"] == 6
assert report["rejected"] == rejected
assert report["comparison"] == comparison
assert json.loads(report_json) == report
assert report_json == json.dumps(report, sort_keys=True, indent=2, allow_nan=False)
print(report_json)
print("Checks passed")


## Reflection

Write 150–250 words addressing the submission questions above. Include one failed prediction or debugging example and explain how you resolved it.

**Your response:**